# Redrob Candidate Ranking — Sandbox Demo (upload-based)

Reproduces the **ranking step** on a small candidate sample, CPU-only, under 5 minutes, **without cloning** (the repo is private).

**How to run:**
1. On your machine, zip the repo into `repo.zip`.
2. Run **Cell 1** and upload `repo.zip` when prompted.
3. Run the remaining cells top to bottom.

Full 100K reproduction happens in the organizers' Stage-3 environment from the GitHub repo (access granted at Stage 3). This is the small-sample check required by submission_spec Section 10.5.

## 1. Upload the repo zip

In [1]:
import zipfile, os, shutil
from google.colab import files

# clean any previous extraction
shutil.rmtree('repo', ignore_errors=True)

uploaded = files.upload()  # pick the NEW repo.zip
zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as z:
    z.extractall('repo')

# find the folder containing BOTH rank.py and artifacts/ (handles any nesting)
target = None
for root, dirs, fnames in os.walk('repo'):
    if 'rank.py' in fnames and 'artifacts' in dirs:
        target = root
        break

assert target, "could not find a folder with both rank.py and artifacts/ — is this the corrected zip?"
os.chdir(target)

print("working directory:", os.getcwd())
print("top-level contents:", sorted(os.listdir('.')))
print("artifacts/:", sorted(os.listdir('artifacts'))[:8])
print("\nparquet exists:", os.path.isfile('artifacts/features_100k.parquet'))

Saving repo.zip to repo (2).zip
working directory: /content/repo
top-level contents: ['artifacts', 'common', 'rank.py', 'requirements.txt', 'sample_candidates.json', 'stage5', 'stage7']
artifacts/: ['audit_log.json', 'blend.json', 'composite_weights.json', 'engine_choice.json', 'features_100k.parquet', 'lgb_features.json', 'ranker_llm.txt', 'ranker_rule.txt']

parquet exists: True


## 2. Install dependencies (CPU)

In [2]:
# Install only what the ranking step needs. Do NOT pin numpy/pandas —
# Colab ships numpy 2.x and downgrading mid-session breaks binary compatibility.
# rank.py is version-agnostic, so we use Colab’s existing numpy/pandas.
!pip install -q lightgbm pyarrow

## 3. Build a 100-candidate sample

In [4]:
import os, glob
print("current dir:", os.getcwd())
print("contents here:", os.listdir('.'))
print("\nrank.py locations:", glob.glob('**/rank.py', recursive=True))
print("parquet locations:", glob.glob('**/features_100k.parquet', recursive=True))

current dir: /content/repo
contents here: ['common', 'requirements.txt', 'sample_candidates.json', 'artifacts', 'stage7', 'stage5', 'rank.py']

rank.py locations: ['rank.py']
parquet locations: ['artifacts/features_100k.parquet']


In [5]:
import json, pandas as pd
df = pd.read_parquet('artifacts/features_100k.parquet')
sample_ids = list(df.index[:100])
try:
    full = {c['candidate_id']: c for c in json.load(open('sample_candidates.json'))}
except Exception:
    full = {}
with open('sandbox_candidates.jsonl', 'w') as f:
    for cid in sample_ids:
        rec = full.get(cid, {'candidate_id': cid, 'profile': {}, 'career_history': [], 'skills': [], 'redrob_signals': {}})
        f.write(json.dumps(rec) + '\n')
print('wrote', len(sample_ids), 'candidates')

wrote 100 candidates


## 4. Run the ranking step (CPU, < 5 min)

In [6]:
import time
t0 = time.time()
!python rank.py --candidates sandbox_candidates.jsonl --out sandbox_submission.csv
print(f'\nelapsed: {time.time()-t0:.1f}s')

REPO_ROOT = /content/repo
scoring 100 candidates
submission -> sandbox_submission.csv
top 5: ['CAND_0000031', 'CAND_0000063', 'CAND_0000025', 'CAND_0000048', 'CAND_0000060']
elapsed: 5.0s  (limit 300s)
STAGE 7.72 (rank.py): PASS

elapsed: 5.6s


## 5. Show the ranked output

In [7]:
import pandas as pd
out = pd.read_csv('sandbox_submission.csv')
print('rows:', len(out), '| distinct reasoning:', out.reasoning.nunique())
out.head(15)

rows: 100 | distinct reasoning: 55


,candidate_id,rank,score,reasoning
0,CAND_0000031,1,0.9990,Recommendation Systems Engineer at Swiggy (6y)...
1,CAND_0000063,2,0.9894,(0y). covers JD requirements: production Pyth...
2,CAND_0000025,3,0.9798,Frontend Engineer at Tech Mahindra (7y). Built...
3,CAND_0000048,4,0.9702,Mobile Developer at CRED (10y). Built and main...
4,CAND_0000060,5,0.9606,(0y). covers JD requirements: production Pyth...
5,CAND_0000088,6,0.9510,(0y). covers JD requirements: production Pyth...
6,CAND_0000055,7,0.9414,(0y). covers JD requirements: production Pyth...
7,CAND_0000038,8,0.9318,Java Developer at Swiggy (7y). covers JD requi...
8,CAND_0000019,9,0.9222,Project Manager at Wayne Enterprises (6y). Str...
9,CAND_0000014,10,0.9126,Frontend Engineer at Zomato (8y). Built and ma...
